In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.delhi_air.silver_pm25 AS
WITH api AS (
    SELECT sensor_id, ingested_at,
           CAST(get_json_object(raw_json, '$.value') AS DOUBLE) AS pm25_ugm3,
           get_json_object(raw_json, '$.period.interval') AS interval_text,
           CAST(get_json_object(raw_json, '$.period.datetimeFrom.utc') AS TIMESTAMP) AS period_start_utc,
           CAST(get_json_object(raw_json, '$.period.datetimeTo.utc') AS TIMESTAMP) AS period_end_utc,
           'api' AS source
    FROM workspace.delhi_air.bronze_openaq_measurements
),
sensor_interval AS (
    SELECT sensor_id, max(interval_text) AS interval_text
    FROM api
    GROUP BY sensor_id
),
archive AS (
    SELECT a.sensor_id, a.ingested_at, a.value AS pm25_ugm3, i.interval_text,
           CASE WHEN i.interval_text = '01:00:00'
                THEN to_timestamp(a.datetime_text, "yyyy-MM-dd'T'HH:mm:ssXXX") - INTERVAL 60 MINUTES
                ELSE to_timestamp(a.datetime_text, "yyyy-MM-dd'T'HH:mm:ssXXX") - INTERVAL 15 MINUTES
           END AS period_start_utc,
           to_timestamp(a.datetime_text, "yyyy-MM-dd'T'HH:mm:ssXXX") AS period_end_utc,
           'archive' AS source
    FROM workspace.delhi_air.bronze_openaq_archive a
    JOIN sensor_interval i ON a.sensor_id = i.sensor_id
),
combined AS (
    SELECT * FROM api
    UNION ALL
    SELECT * FROM archive
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY sensor_id, period_start_utc
               ORDER BY CASE WHEN source = 'archive' THEN 0 ELSE 1 END, ingested_at DESC
           ) AS rn
    FROM combined
)
SELECT k.sensor_id, r.location_id, r.location_name, r.provider,
       k.period_start_utc, k.period_end_utc, k.interval_text, k.pm25_ugm3, k.source
FROM ranked k
JOIN workspace.delhi_air.sensor_registry r ON k.sensor_id = r.sensor_id
WHERE k.rn = 1
""")
print("done")